# MANTICORE V7.3.1.4 — Download & Archive Integrity Audit

Independent test: **does not launch training, does not modify main checkpoints**, and does **not** automatically download years of data.

1. Verify Binance Vision `.CHECKSUM` reachability for BTC COIN-M and ETH/SOL USD-M.
2. Optional full monthly ZIP download with SHA-256, CRC, CSV schema and sample parsing.
3. Inspect locally available HistData NSX/USD monthly ZIPs; the main notebook **does not have an NSX auto-downloader**.
4. Save JSON audit report inside this notebook's isolated folder.

Run on Colab with internet connectivity; no GPU needed.


In [ ]:
import hashlib,io,json,os,re,time,urllib.request,urllib.error,zipfile
from pathlib import Path
from datetime import datetime,timezone
import pandas as pd
ROOT=Path('/content/manticore_v7314_download_audit')
ROOT.mkdir(parents=True,exist_ok=True)
# Configure a known past month. A single full monthly aggTrades ZIP may be very large.
TEST_MONTH='2025-01'
DOWNLOAD_FULL_MONTH=False  # set True only when ready for multi-GB downloads
NSX_DIR=Path('/content/drive/MyDrive/HERMES_Colab/ai_training/histdata_tick/NSXUSD')
ASSETS=[('BTCUSD','futures/cm','BTCUSD_PERP'),
        ('ETHUSDT','futures/um','ETHUSDT'),
        ('SOLUSDT','futures/um','SOLUSDT')]
UA={'User-Agent':'MANTICORE-research/1.0'}
report={'timestamp':datetime.now(timezone.utc).isoformat(),'month':TEST_MONTH,
        'download_full_month':DOWNLOAD_FULL_MONTH,'binance':{},'histdata':{}}
def url_for(market,symbol,month):
    return f'https://data.binance.vision/data/{market}/monthly/aggTrades/{symbol}/{symbol}-aggTrades-{month}.zip'
def get_text(url,max_bytes=1024):
    with urllib.request.urlopen(urllib.request.Request(url,headers=UA),timeout=30) as r:
        return r.status,r.read(max_bytes).decode('ascii','replace')
def sha256(path):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(2*1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()
print('Audit location:',ROOT,'Full monthly downloads enabled:',DOWNLOAD_FULL_MONTH)

In [ ]:
# Test remote checksum availability independently of entire archives.
for name,market,symbol in ASSETS:
    url=url_for(market,symbol,TEST_MONTH)
    item={'url':url,'checksum_url':url+'.CHECKSUM'}
    try:
        t0=time.monotonic()
        status,body=get_text(url+'.CHECKSUM')
        expected=(body.strip().split() or [''])[0].lower()
        assert status==200 and re.fullmatch('[0-9a-f]{64}',expected),f'bad checksum response: {body[:100]}'
        item.update(status='CHECKSUM_OK',sha256_expected=expected,latency_seconds=round(time.monotonic()-t0,2))
    except Exception as e:
        item.update(status='ERROR',error=f'{type(e).__name__}: {e}')
    report['binance'][name]=item
    print(name,item['status'],item.get('error',''))


In [ ]:
# Optional actual ZIP transfer + checksum/CRC + representative rows.
# Store in isolated ephemeral folder, never main archive folder.
if DOWNLOAD_FULL_MONTH:
    for name,market,symbol in ASSETS:
        item=report['binance'][name]
        if item['status']!='CHECKSUM_OK':
            print(name,'SKIP: checksum endpoint failed');continue
        url=item['url'];dst=ROOT/f'{symbol}-aggTrades-{TEST_MONTH}.zip'
        partial=dst.with_suffix('.part')
        try:
            t0=time.monotonic()
            req=urllib.request.Request(url,headers=UA)
            with urllib.request.urlopen(req,timeout=120) as r, open(partial,'wb') as f:
                while (chunk:=r.read(2*1024*1024)):f.write(chunk)
            assert sha256(partial)==item['sha256_expected'],'SHA256 mismatch'
            assert zipfile.is_zipfile(partial),'Not ZIP'
            with zipfile.ZipFile(partial) as archive:
                assert archive.testzip() is None,'CRC failure'
                names=[x for x in archive.namelist() if x.lower().endswith('.csv')]
                assert len(names)==1,f'Unexpected CSV members: {names}'
                with archive.open(names[0]) as stream:
                    first_lines=[stream.readline().decode('utf-8-sig','replace').strip() for _ in range(3)]
            partial.replace(dst)
            item.update(status='FULL_ZIP_OK',bytes=dst.stat().st_size,
                        seconds=round(time.monotonic()-t0,2),csv_sample=first_lines)
        except Exception as e:
            item.update(status='FULL_ZIP_ERROR',error=f'{type(e).__name__}: {e}')
        finally:
            partial.unlink(missing_ok=True)
        print(name,item['status'],item.get('error',''),item.get('bytes'))
else:
    print('FULL ZIP test disabled. Only .CHECKSUM endpoints probed; no assertion of downloaded data.')


In [ ]:
# HistData's NSX monthly ZIP files need to be supplied in Drive.
# Mount Drive if desired before running this cell.
zips=sorted(NSX_DIR.glob('*.zip')) if NSX_DIR.exists() else []
nsx={'directory':str(NSX_DIR),'zip_count':len(zips),'files_checked':[]}
for f in zips[:12]:
    rec={'name':f.name,'bytes':f.stat().st_size}
    try:
        with zipfile.ZipFile(f) as archive:
            bad=archive.testzip()
            names=[x for x in archive.namelist() if x.lower().endswith('.csv')]
            assert bad is None and len(names)==1,f'CRC failure or CSV count={len(names)}'
            with archive.open(names[0]) as stream:
                first=stream.readline().decode('utf-8-sig','replace').strip()
            rec.update(status='ZIP_OK',csv_member=names[0],sample_line=first)
    except Exception as e:
        rec.update(status='ERROR',error=f'{type(e).__name__}: {e}')
    nsx['files_checked'].append(rec)
nsx['status']='NOT_AVAILABLE' if not zips else 'SAMPLE_ZIPS_AUDITED'
report['histdata']=nsx
print('NSX:',nsx['status'],'ZIP files:',len(zips))


In [ ]:
path=ROOT/'download_audit.json'
path.write_text(json.dumps(report,indent=2,ensure_ascii=False))
print('Audit report:',path)
print('Binance statuses:',{k:v['status'] for k,v in report['binance'].items()})
print('HistData status:',report['histdata']['status'])
print('NOTE: CHECKSUM_OK proves that a checksum URL responded, not that the ZIP was downloaded or parsed.')
